In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
from math import log, sqrt, exp
from scipy.stats import norm
import yfinance as yf

In [40]:
#Loading the data
df_27 = pd.read_csv(r'C:\Users\VIDUR\Desktop\PYTHON\My_Project\27-1.csv')
df_28 = pd.read_csv(r'C:\Users\VIDUR\Desktop\PYTHON\My_Project\28-1.csv')
df_29 = pd.read_csv(r'C:\Users\VIDUR\Desktop\PYTHON\My_Project\29-1.csv')

In [41]:
# Data manipulation 
#Adding time to maturity column
expiry = pd.Timestamp("2026-01-30")
df_27["T"] = (expiry - pd.to_datetime("2026-01-27")).days / 252
df_28["T"] = (expiry - pd.to_datetime("2026-01-28")).days / 252
df_29["T"] = (expiry - pd.to_datetime("2026-01-29")).days / 252

# Adding spot to the DataFrame

meta = yf.Ticker("META")

data = meta.history(start="2026-01-01", end="2026-02-01")

spot_27 = data.loc["2026-01-27"]["Close"]
spot_28 = data.loc["2026-01-28"]["Close"]
spot_29 = data.loc["2026-01-29"]["Close"]

df_27["Spot"] = spot_27
df_28["Spot"] = spot_28
df_29["Spot"] = spot_29

print(df_27.columns)

#Adding moneyness 
for df in [df_27, df_28, df_29]:
    df["moneyness"] = df["Spot"] / df["Strike"]
    df["log_moneyness"] = np.log(df["Spot"] / df["Strike"])



Index(['Strike', 'Ticker', 'Bid', 'Ask', 'Last', 'IVM', 'Vol', 'Unnamed: 7',
       'Strike.1', 'Ticker.1', 'Bid.1', 'Ask.1', 'Last.1', 'IVM.1', 'Vol.1',
       'T', 'Spot'],
      dtype='str')


In [ ]:
#defining functions to calculate d1 and d2 for the Black-Scholes model

def _d1_d2(S, K, T, r, sigma):
    d1 = (log(S/K) + (r + (sigma**2)/2)*T) / (sigma*sqrt(T))
    d2 = d1 - sigma*sqrt(T)
    return d1, d2

def BS_model(S, K, T, r, sigma, option_type = 'call'):
    if option_type == 'call':
        d1, d2 = _d1_d2(S, K, T, r, sigma)
        option_price = S*norm.cdf(d1) - K*exp(-r*T)*norm.cdf(d2)
    elif option_type == 'put':
        d1, d2 = _d1_d2(S, K, T, r, sigma)
        option_price = K*exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)
    else:
        raise ValueError("Invalid option type. Use 'call' or 'put'.")
    return option_price

def bs_theta(S, K, r, sigma, T, option_type='call'):
    
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    term1 = -(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))

    if option_type == 'call':
        theta = term1 - r * K * np.exp(-r * T) * norm.cdf(d2)
    else:
        theta = term1 + r * K * np.exp(-r * T) * norm.cdf(-d2)

    return theta

def theta_surface_model(option_type='call'):
    theta_values = []

    for index, row in df_27.iterrows():
        S = row['Spot']
        K = row['Strike']
        T = row['T']
        r = row['Risk Free Rate']
        sigma = row['Volatility']

        theta = bs_theta(S, K, r, sigma, T, option_type)

        theta_values.append(theta / 365)

    return theta_values

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 1 — Constants & Spot Prices
# ─────────────────────────────────────────────────────────────────
from scipy.interpolate import griddata
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm

R = 0.0425          # ~4.25% risk-free rate (Fed Funds, Jan 2026)
EXPIRY = pd.Timestamp('2026-01-30')

# META closing prices sourced from yfinance (same approach as notebook)
SPOTS = {
    '2026-01-27': spot_27,
    '2026-01-28': spot_28,
    '2026-01-29': spot_29,
}

# Days to expiry (calendar days)
DTE_MAP = {'2026-01-27': 3, '2026-01-28': 2, '2026-01-29': 1}


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 2 — Parse calls & puts from each CSV into a unified DataFrame
# ─────────────────────────────────────────────────────────────────
def parse_options(df_raw, date_str):
    """Split the side-by-side call/put CSV layout into long-format rows."""
    spot = SPOTS[date_str]
    T_cal = (EXPIRY - pd.Timestamp(date_str)).days / 365

    calls = df_raw[['Strike', 'IVM']].copy()
    calls.columns = ['Strike', 'IVM']
    calls['Type'] = 'call'

    puts = df_raw[['Strike.1', 'IVM.1']].copy()
    puts.columns = ['Strike', 'IVM']
    puts['Type'] = 'put'

    out = pd.concat([calls, puts], ignore_index=True)
    out = out.dropna(subset=['Strike'])
    out = out[out['IVM'] > 5]           # filter zero / stale quotes
    out['Sigma']        = out['IVM'] / 100.0
    out['T']            = T_cal
    out['Spot']         = spot
    out['Date']         = date_str
    out['DTE']          = DTE_MAP[date_str]
    out['LogMoneyness'] = np.log(spot / out['Strike'])
    return out

opts_27 = parse_options(df_27, '2026-01-27')
opts_28 = parse_options(df_28, '2026-01-28')
opts_29 = parse_options(df_29, '2026-01-29')
all_opts = pd.concat([opts_27, opts_28, opts_29], ignore_index=True)
print(all_opts.groupby(['Date','Type']).size())


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 3 — Compute Theta for every option row
# ─────────────────────────────────────────────────────────────────
def bs_theta(S, K, r, sigma, T, option_type='call'):
    """Black-Scholes theta, returned as $/calendar-day (divided by 365)."""
    if T <= 0 or sigma <= 0:
        return np.nan
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    term1 = -(S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T))
    if option_type == 'call':
        theta = term1 - r * K * np.exp(-r * T) * norm.cdf(d2)
    else:
        theta = term1 + r * K * np.exp(-r * T) * norm.cdf(-d2)
    return theta / 365   # per calendar day

all_opts['Theta'] = all_opts.apply(
    lambda row: bs_theta(row['Spot'], row['Strike'], R,
                         row['Sigma'], row['T'], row['Type']), axis=1
)
all_opts = all_opts.dropna(subset=['Theta'])
print(all_opts[['Date','Type','Strike','Sigma','DTE','Theta']].head(10).to_string(index=False))


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 4 — 3-D Theta Surface (Log-Moneyness axis)
# ─────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(18, 8))
fig.patch.set_facecolor('#0d1117')

for idx, (opt_type, title_suffix) in enumerate([('call','Calls'),('put','Puts')]):
    sub = all_opts[all_opts['Type'] == opt_type].copy()
    ax  = fig.add_subplot(1, 2, idx+1, projection='3d')
    ax.set_facecolor('#0d1117')

    dte_v = sub['DTE'].values
    lm_v  = sub['LogMoneyness'].values   # ln(S/K): ATM=0, +ve=ITM, -ve=OTM
    th_v  = sub['Theta'].values

    DTE_G, LM_G = np.meshgrid(
        np.linspace(dte_v.min(), dte_v.max(), 60),
        np.linspace(lm_v.min(),  lm_v.max(),  60)
    )
    TH_G = griddata((dte_v, lm_v), th_v, (DTE_G, LM_G), method='cubic')

    valid = TH_G[~np.isnan(TH_G)]
    norm_th = (TH_G - valid.min()) / (valid.max() - valid.min())
    cmap = cm.RdYlGn_r

    ax.plot_surface(DTE_G, LM_G, TH_G, facecolors=cmap(norm_th),
                    alpha=0.90, linewidth=0, antialiased=True)
    ax.scatter(dte_v, lm_v, th_v, c=th_v, cmap='RdYlGn_r', s=20, zorder=5)

    # Cyan dashed line tracing ATM (log-moneyness ≈ 0) across DTEs
    dte_line = np.array([1, 2, 3])
    atm_theta = []
    for d in dte_line:
        sub_d   = sub[sub['DTE'] == d]
        atm_row = sub_d.iloc[sub_d['LogMoneyness'].abs().argsort()[:1]]
        atm_theta.append(atm_row['Theta'].values[0])
    ax.plot(dte_line, [0,0,0], atm_theta,
            color='cyan', lw=2.5, ls='--', zorder=10)

    ax.set_xlabel('Days to Expiry',        color='white', labelpad=8)
    ax.set_ylabel('Log-Moneyness ln(S/K)', color='white', labelpad=10)
    ax.set_zlabel('Theta ($/day)',          color='white', labelpad=8)
    ax.set_title(f'META Theta Surface — {title_suffix}\n(Expiry Jan 30, 2026)',
                 color='white', fontsize=12, fontweight='bold', pad=12)
    ax.set_xticks([1,2,3])
    ax.set_xticklabels(['1d','2d','3d'], color='#aaa')
    ax.tick_params(colors='#aaa')
    for pane in [ax.xaxis.pane, ax.yaxis.pane, ax.zaxis.pane]:
        pane.fill = False; pane.set_edgecolor('#333')
    ax.grid(color='#333', linewidth=0.5)
    ax.view_init(elev=28, azim=-55)

    sm = cm.ScalarMappable(cmap=cmap); sm.set_array(th_v)
    cb = fig.colorbar(sm, ax=ax, shrink=0.5, pad=0.1)
    cb.set_label('Theta ($/day)', color='white', fontsize=8)
    cb.ax.yaxis.set_tick_params(color='white')
    plt.setp(cb.ax.get_yticklabels(), color='white', fontsize=7)

plt.suptitle('META Options — Theta Decay Surface | Jan 27–29, 2026\n'
             'Y-axis: Log-Moneyness ln(S/K) — ATM=0, +ve=ITM, −ve=OTM',
             color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('theta_surface_3d.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 5 — Cross-sections: Theta vs Log-Moneyness by date
# ─────────────────────────────────────────────────────────────────
fig2, axes = plt.subplots(1, 3, figsize=(16, 5))
fig2.patch.set_facecolor('#0d1117')
colors_map = {'call': '#00bfff', 'put': '#ff6b6b'}

for i, (date, dte) in enumerate(sorted(DTE_MAP.items(), key=lambda x: -x[1])):
    ax  = axes[i]
    ax.set_facecolor('#1a1f2e')
    sub = all_opts[all_opts['Date'] == date]

    for opt_type in ['call', 'put']:
        s = sub[sub['Type']==opt_type].sort_values('LogMoneyness')
        ax.plot(s['LogMoneyness'], s['Theta'],
                color=colors_map[opt_type], lw=2,
                label=opt_type.capitalize(), marker='o', ms=3)

    ax.axvline(0, color='yellow', lw=1.5, ls='--', alpha=0.85, label='ATM')
    ax.axhline(0, color='#555', lw=0.8)
    ax.axvspan( 0,  0.12, alpha=0.06, color='green')
    ax.axvspan(-0.12, 0,  alpha=0.06, color='red')

    ax.set_title(f'{date[5:]} | DTE={dte}d', color='white', fontsize=11, fontweight='bold')
    ax.set_xlabel('Log-Moneyness  ln(S/K)', color='#aaa', fontsize=9)
    ax.set_ylabel('Theta ($/day)',           color='#aaa', fontsize=9)
    ax.tick_params(colors='#aaa')
    ax.legend(fontsize=7, facecolor='#1a1f2e', edgecolor='#555', labelcolor='white')
    ax.grid(color='#333', lw=0.5, alpha=0.7)
    for sp in ax.spines.values(): sp.set_edgecolor('#444')

plt.suptitle('Theta vs Log-Moneyness | META Jan 2026 Expiry',
             color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('theta_cross_sections.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────
# CELL 6 — Summary Statistics
# ─────────────────────────────────────────────────────────────────
summary = all_opts.groupby(['Date','Type'])['Theta'].agg(['min','max','mean']).round(4)
print('\n── Theta Summary ($/calendar day) ──')
print(summary.to_string())

# ATM theta (closest strike to spot)
print('\n── ATM Theta ──')
for date in sorted(DTE_MAP.keys(), reverse=True):
    spot = DATE_SPOTS[date]
    sub  = all_opts[all_opts['Date']==date]
    for ot in ['call','put']:
        s = sub[sub['Type']==ot]
        atm_row = s.iloc[(s['Strike']-spot).abs().argsort()[:1]]
        theta_atm = atm_row['Theta'].values[0]
        k_atm     = atm_row['Strike'].values[0]
        dte       = DTE_MAP[date]
        print(f'  {date} | DTE={dte}d | {ot:4s} | ATM K={k_atm:.1f} | Theta={theta_atm:.4f} $/day')
